In [1]:
import sys
import pandas as pd

# Trỏ đường dẫn về thư mục gốc
sys.path.append('..')

from src.data.loader import fetch_and_load_data
from src.data.cleaner import clean_and_preprocess
from src.mining.association import mine_failure_rules
from src.mining.clustering import cluster_machine_behavior

# 1. Nạp dữ liệu
df_raw = fetch_and_load_data()
df_processed = clean_and_preprocess(df_raw)

# 2. Chạy Apriori tìm luật kết hợp lỗi
print("--- TOP CÁC TỔ HỢP ĐIỀU KIỆN GÂY LỖI MÁY (APRIORI) ---")
rules = mine_failure_rules(df_raw)
display(rules.head(5))

# 3. Chạy Phân cụm KMeans
print("\n--- PHÂN CỤM HÀNH VI MÁY MÓC (CLUSTERING) ---")
df_clustered, sil_score, kmeans_model = cluster_machine_behavior(df_processed)
print(f"Chỉ số Silhouette (đánh giá chất lượng cụm): {sil_score:.3f}")

# 4. Profiling: Đặc điểm của từng cụm để lên lịch bảo trì
print("\nHồ sơ các cụm (Trung bình các thông số đã chuẩn hóa):")
cluster_profile = df_clustered.groupby('Behavior_Cluster')[['Air temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]']].mean()
display(cluster_profile)

Dữ liệu đã tồn tại trong data/raw/.
--- TOP CÁC TỔ HỢP ĐIỀU KIỆN GÂY LỖI MÁY (APRIORI) ---


,antecedents,consequents,support,confidence,lift
27,"(Speed_Low, Torque_High, Temp_High)",(Machine_Failure),0.0130,0.226876,6.692510
16,"(Speed_Low, Temp_High)",(Machine_Failure),0.0174,0.187298,5.525020
13,"(Torque_High, Temp_High)",(Machine_Failure),0.0142,0.177722,5.242541
21,"(Speed_Low, Torque_High)",(Machine_Failure),0.0187,0.165049,4.868689
31,"(Torque_High, Temp_High)","(Speed_Low, Machine_Failure)",0.0130,0.162703,6.894211



--- PHÂN CỤM HÀNH VI MÁY MÓC (CLUSTERING) ---
Chỉ số Silhouette (đánh giá chất lượng cụm): 0.262

Hồ sơ các cụm (Trung bình các thông số đã chuẩn hóa):


,Air temperature [K],Rotational speed [rpm],Torque [Nm]
Behavior_Cluster,,,
0,0.066243,1.434295,-1.341387
1,0.857332,-0.388450,0.372180
2,-0.873112,-0.359749,0.327744
